# CIBERSORTx Mixture File Generator — GSE14905 (Healthy vs. Lesional only)

Builds the bulk-sample "mixture" file CIBERSORTx needs, from
[GSE14905](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE14905) (Affymetrix HG-U133_Plus2 / GPL570).

**This version excludes "Uninvolved Skin" (non-lesional) samples by default**, keeping only the
21 healthy and 33 lesional samples (54 total). If you also want to drop the same lesional PCA
outliers excluded in the manuscript, add their GSM IDs to `EXCLUDE_GSM_IDS` below.

**Output:** `GSE14905_cibersortx_mixture_healthy_vs_lesional.txt` — tab-delimited, genes (rows) x
samples (columns), linear-scale, non-negative, unique gene symbols, no missing values. Upload this
alongside your LM22 signature matrix on the
[CIBERSORTx web portal](https://cibersortx.stanford.edu/) (Cell Fractions module).

**To include Uninvolved Skin samples again later:** set `EXCLUDE_CONDITIONS = []` in the config cell.


## 0. Setup

In [ ]:
!pip install -q GEOparse

import GEOparse
import pandas as pd
import numpy as np
import os

OUT_DIR = "/content/cibersortx_input"
os.makedirs(OUT_DIR, exist_ok=True)


## 1. Config — edit these if needed

In [ ]:
GEO_SERIES = "GSE14905"

# If you know the GSM IDs of the 3 lesional PCA outliers excluded in the manuscript, list them here.
# Leave empty to skip GSM-level exclusion.
EXCLUDE_GSM_IDS = []   # e.g. ["GSM123456", "GSM123457", "GSM123458"]

# Condition labels to drop entirely before writing the mixture file.
# Set to [] to keep all sample types (healthy + lesional + Uninvolved Skin).
# Default here excludes non-lesional ("Uninvolved Skin") samples, keeping only healthy + lesional.
EXCLUDE_CONDITIONS = ["Uninvolved Skin"]

# How to collapse multiple probes mapping to the same gene symbol.
# CIBERSORTx accepts either; 'mean' matches what your manuscript used for its own g:Profiler mapping.
PROBE_COLLAPSE_METHOD = "mean"   # "mean" or "max"

OUTPUT_FILENAME = f"{OUT_DIR}/GSE14905_cibersortx_mixture_healthy_vs_lesional.txt"


## 2. Download GSE14905

In [ ]:
print(f"Downloading {GEO_SERIES} (series matrix + platform annotation)...")
gse = GEOparse.get_GEO(geo=GEO_SERIES, destdir=f"{OUT_DIR}/raw")

expr = gse.pivot_samples("VALUE")   # probes x samples
print(f"Raw expression matrix: {expr.shape[0]} probes x {expr.shape[1]} samples")
expr.head()


31-Aug-2026 14:35:13 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE14nnn/GSE14905/soft/GSE14905_family.soft.gz to /content/cibersortx_input/raw/GSE14905_family.soft.gz
INFO:GEOparse:Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE14nnn/GSE14905/soft/GSE14905_family.soft.gz to /content/cibersortx_input/raw/GSE14905_family.soft.gz


100%|██████████| 47.9M/47.9M [00:00<00:00, 61.6MB/s]
31-Aug-2026 14:35:14 DEBUG downloader - Size validation passed
DEBUG:GEOparse:Size validation passed
31-Aug-2026 14:35:14 DEBUG downloader - Moving /tmp/tmp4f6c77sn to /content/cibersortx_input/raw/GSE14905_family.soft.gz
DEBUG:GEOparse:Moving /tmp/tmp4f6c77sn to /content/cibersortx_input/raw/GSE14905_family.soft.gz
31-Aug-2026 14:35:14 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE14nnn/GSE14905/soft/GSE14905_family.soft.gz
DEBUG:GEOparse:Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE14nnn/GSE14905/soft/GSE14905_family.soft.gz
31-Aug-2026 14:35:14 INFO GEOparse - Parsing /content/cibersortx_input/raw/GSE14905_family.soft.gz: 
INFO:GEOparse:Parsing /content/cibersortx_input/raw/GSE14905_family.soft.gz: 
31-Aug-2026 14:35:14 DEBUG GEOparse - DATABASE: GeoMiame
DEBUG:GEOparse:DATABASE: GeoMiame
31-Aug-2026 14:35:14 DEBUG GEOparse - SERIES: GSE14905
DEBUG:GEOparse:SERIES: GSE14905

Raw expression matrix: 54675 probes x 82 samples


name,GSM372286,GSM372287,GSM372288,GSM372289,GSM372290,GSM372291,GSM372292,GSM372293,GSM372294,GSM372295,...,GSM372358,GSM372359,GSM372360,GSM372361,GSM372362,GSM372363,GSM372364,GSM372365,GSM372366,GSM372367
ID_REF,,,,,,,,,,,,,,,,,,,,,
1007_s_at,8.927226,9.323153,9.425702,8.530724,9.813320,9.472235,9.780397,9.898842,9.598552,10.176234,...,8.669118,9.417391,8.505122,10.288123,9.062033,9.765311,8.788144,9.174795,8.864014,9.276448
1053_at,6.806054,6.130473,5.169635,5.915887,5.956715,7.171988,6.746940,7.227279,6.177630,6.326832,...,7.661062,7.305561,7.872603,6.554745,7.740349,6.655057,7.611210,7.306302,7.620674,7.835133
117_at,5.843562,5.842975,4.749800,5.252182,4.986098,6.053810,4.760123,5.337666,4.577448,3.077564,...,6.346169,5.670163,5.423811,4.142631,6.267795,5.338442,5.974267,5.926321,5.760651,5.223836
121_at,5.557898,5.171538,5.547484,5.366430,5.554267,5.570317,5.552489,5.560102,5.828432,5.564013,...,6.313055,5.561764,6.018332,5.657773,6.260822,5.486076,5.561106,5.556993,7.148681,6.120288
1255_g_at,2.104409,2.081902,2.093000,2.092062,2.081100,2.174821,2.106389,2.119007,2.162319,2.119663,...,2.074408,2.084942,2.089481,2.091197,2.101904,2.074014,2.074417,2.097319,2.113838,2.090939


## 3. Check scale — linear vs. log2

If the max value across the matrix is well under ~30, the data is almost certainly log2-transformed
(microarray intensities in linear space commonly range into the tens of thousands). CIBERSORTx expects
linear scale, so this cell un-logs the data (`2^x`) when that pattern is detected, and tells you which
path it took rather than assuming silently.


In [ ]:
max_val = expr.values.max()
min_val = expr.values.min()
print(f"Value range across matrix: min={min_val:.3f}, max={max_val:.3f}")

if max_val < 30:
    print("\n--> Data appears LOG2-transformed. Un-logging (2^x) to get linear scale for CIBERSORTx.")
    expr_linear = 2 ** expr
else:
    print("\n--> Data appears to already be in LINEAR scale. Using as-is.")
    expr_linear = expr.copy()

# CIBERSORTx requires non-negative values
n_negative = (expr_linear < 0).sum().sum()
if n_negative > 0:
    print(f"WARNING: {n_negative} negative values found after transformation -- clipping to 0.")
    expr_linear = expr_linear.clip(lower=0)

print(f"\nFinal value range: min={expr_linear.values.min():.2f}, max={expr_linear.values.max():.2f}")


Value range across matrix: min=0.782, max=15.956

--> Data appears LOG2-transformed. Un-logging (2^x) to get linear scale for CIBERSORTx.

Final value range: min=1.72, max=63578.91


## 4. Sample metadata — condition labels

**Note:** GEO submitters for GSE14905 used the label **"Uninvolved Skin"** for non-lesional samples,
not "non-lesional" — the classifier below checks for both phrasings so nothing falls through to
`"unknown"`.


In [ ]:
sample_meta = pd.DataFrame({
    "title": {gsm: s.metadata["title"][0] for gsm, s in gse.gsms.items()},
})

# Extract condition from sample title text.
# GSE14905 specifically uses "Uninvolved Skin" for non-lesional samples (not "non-lesional"),
# so both phrasings are checked to avoid samples falling through to 'unknown'.
def classify_condition(title):
    t = title.lower()
    if "uninvolved" in t or ("non" in t and "lesion" in t):
        return "Uninvolved Skin"
    elif "lesion" in t:
        return "lesional"
    elif "normal" in t or "healthy" in t or "control" in t:
        return "healthy"
    return "unknown"

sample_meta["condition"] = sample_meta["title"].apply(classify_condition)
print(sample_meta["condition"].value_counts())

n_unknown = (sample_meta["condition"] == "unknown").sum()
if n_unknown > 0:
    print(f"\nWARNING: {n_unknown} sample(s) could not be classified. Inspect their titles below --")
    print("GEO submitters don't always use consistent wording, so a new phrasing may need adding")
    print("to classify_condition().")
    print(sample_meta.loc[sample_meta["condition"] == "unknown", "title"])

sample_meta.to_csv(f"{OUT_DIR}/GSE14905_sample_metadata.csv")
sample_meta.head(10)


condition
lesional           33
Uninvolved Skin    28
healthy            21
Name: count, dtype: int64


,title,condition
GSM372286,"Normal Skin, Normal-1",healthy
GSM372287,"Normal Skin, Normal-2",healthy
GSM372288,"Normal Skin, Normal-3",healthy
GSM372289,"Normal Skin, Normal-4",healthy
GSM372290,"Normal Skin, Normal-5",healthy
GSM372291,"Normal Skin, Normal-6",healthy
GSM372292,"Normal Skin, Normal-7",healthy
GSM372293,"Normal Skin, Normal-8",healthy
GSM372294,"Normal Skin, Normal-9",healthy
GSM372295,"Normal Skin, Normal-10",healthy


In [ ]:
# Apply condition-based exclusion first (e.g. dropping Uninvolved Skin samples)
if EXCLUDE_CONDITIONS:
    gsms_to_drop_by_condition = sample_meta.index[sample_meta["condition"].isin(EXCLUDE_CONDITIONS)]
    before = expr_linear.shape[1]
    expr_linear = expr_linear.drop(columns=gsms_to_drop_by_condition, errors="ignore")
    sample_meta = sample_meta.drop(index=gsms_to_drop_by_condition, errors="ignore")
    print(f"Excluded {before - expr_linear.shape[1]} sample(s) matching condition(s) {EXCLUDE_CONDITIONS}.")
    print(f"Remaining: {expr_linear.shape[1]} samples")
    print(sample_meta["condition"].value_counts())
else:
    print("No condition-based exclusion applied (EXCLUDE_CONDITIONS is empty).")

# Then apply any explicit GSM-ID exclusion (e.g. known PCA outliers)
if EXCLUDE_GSM_IDS:
    before = expr_linear.shape[1]
    expr_linear = expr_linear.drop(columns=[g for g in EXCLUDE_GSM_IDS if g in expr_linear.columns])
    sample_meta = sample_meta.drop(index=[g for g in EXCLUDE_GSM_IDS if g in sample_meta.index])
    print(f"\nExcluded {before - expr_linear.shape[1]} sample(s) by GSM ID. Remaining: {expr_linear.shape[1]}")
else:
    print("\nNo GSM-ID-based exclusion applied (EXCLUDE_GSM_IDS is empty).")

print(f"\nFinal sample set for mixture file: {expr_linear.shape[1]} samples")


Excluded 28 sample(s) matching condition(s) ['Uninvolved Skin'].
Remaining: 54 samples
condition
lesional    33
healthy     21
Name: count, dtype: int64

No GSM-ID-based exclusion applied (EXCLUDE_GSM_IDS is empty).

Final sample set for mixture file: 54 samples


## 5. Probe -> gene symbol mapping (GPL570 annotation)

In [ ]:
gpl = list(gse.gpls.values())[0]
gpl_table = gpl.table
print("Platform annotation columns available:", list(gpl_table.columns))

gene_col_candidates = [c for c in gpl_table.columns if "gene symbol" in c.lower() or c.lower() == "symbol"]

if not gene_col_candidates:
    raise ValueError(
        "Could not auto-detect a gene symbol column in the GPL570 annotation table. "
        f"Available columns were: {list(gpl_table.columns)}. "
        "Inspect gpl_table manually and set gene_col to the correct column name."
    )

gene_col = gene_col_candidates[0]
print(f"Using annotation column: '{gene_col}'")

probe_to_gene = gpl_table.set_index("ID")[gene_col]
probe_to_gene = probe_to_gene.dropna()
probe_to_gene = probe_to_gene[probe_to_gene.astype(str).str.strip() != ""]

print(f"Probe-to-gene mapping covers {len(probe_to_gene)} probes.")


Platform annotation columns available: ['ID', 'GB_ACC', 'SPOT_ID', 'Species Scientific Name', 'Annotation Date', 'Sequence Type', 'Sequence Source', 'Target Description', 'Representative Public ID', 'Gene Title', 'Gene Symbol', 'ENTREZ_GENE_ID', 'RefSeq Transcript ID', 'Gene Ontology Biological Process', 'Gene Ontology Cellular Component', 'Gene Ontology Molecular Function']
Using annotation column: 'Gene Symbol'
Probe-to-gene mapping covers 45782 probes.


## 6. Collapse to unique gene symbols and write the CIBERSORTx mixture file

In [ ]:
expr_annotated = expr_linear.copy()
expr_annotated["gene"] = expr_annotated.index.map(probe_to_gene)
expr_annotated = expr_annotated.dropna(subset=["gene"])

# Some GPL570 annotation rows list multiple gene symbols separated by '///' for cross-hybridizing probes;
# these are ambiguous and standard practice is to drop them for signature-matrix-based deconvolution.
n_before = expr_annotated.shape[0]
expr_annotated = expr_annotated[~expr_annotated["gene"].astype(str).str.contains("///")]
print(f"Dropped {n_before - expr_annotated.shape[0]} ambiguous multi-gene probes.")

if PROBE_COLLAPSE_METHOD == "mean":
    mixture = expr_annotated.groupby("gene").mean(numeric_only=True)
elif PROBE_COLLAPSE_METHOD == "max":
    # Collapse by selecting, for each gene, the probe row with the highest mean expression across samples
    expr_annotated["_mean_expr"] = expr_annotated.drop(columns="gene").mean(axis=1)
    mixture = (
        expr_annotated.sort_values("_mean_expr", ascending=False)
        .drop_duplicates(subset="gene", keep="first")
        .drop(columns="_mean_expr")
        .set_index("gene")
    )
else:
    raise ValueError("PROBE_COLLAPSE_METHOD must be 'mean' or 'max'")

# Final checks CIBERSORTx cares about
n_na = mixture.isna().sum().sum()
if n_na > 0:
    print(f"WARNING: {n_na} missing values remain -- dropping affected genes.")
    mixture = mixture.dropna()

n_negative = (mixture < 0).sum().sum()
if n_negative > 0:
    print(f"WARNING: {n_negative} negative values remain -- clipping to 0.")
    mixture = mixture.clip(lower=0)

mixture.index.name = "GeneSymbol"
print(f"\nFinal mixture matrix: {mixture.shape[0]} unique genes x {mixture.shape[1]} samples")
mixture.head()


Dropped 2796 ambiguous multi-gene probes.

Final mixture matrix: 21655 unique genes x 54 samples


name,GSM372286,GSM372287,GSM372288,GSM372289,GSM372290,GSM372291,GSM372292,GSM372293,GSM372294,GSM372295,...,GSM372352,GSM372354,GSM372356,GSM372358,GSM372360,GSM372362,GSM372364,GSM372365,GSM372366,GSM372367
GeneSymbol,,,,,,,,,,,,,,,,,,,,,
A1BG,4.843981,8.086500,4.594892,4.595631,4.590429,4.576177,4.603877,4.628443,4.664287,4.658028,...,4.631626,4.796161,4.540295,4.564086,4.562363,4.599016,4.561754,4.552587,4.636484,4.602521
A1BG-AS1,7.193065,12.779944,7.183169,8.957272,7.159781,8.342397,7.193716,7.425345,13.443045,7.750749,...,7.431223,7.377438,6.794644,7.258675,7.036253,10.578100,6.646277,6.976117,7.343439,7.113739
A1CF,5.918883,5.879828,6.170653,6.137120,5.903610,6.891796,6.082273,6.300251,7.225909,7.594716,...,6.347291,6.333315,6.035531,5.865721,6.062225,6.182758,5.708704,6.089952,7.838298,6.200975
A2M,1020.306972,1265.268474,984.615164,2355.413795,670.177168,505.435006,589.919319,459.303334,1371.530338,546.708000,...,492.896260,126.649434,523.659042,400.586265,627.467780,432.469518,459.610888,587.799841,292.232180,371.595412
A2M-AS1,8.763681,9.076776,8.722920,24.852238,14.351854,9.171511,8.622114,8.674482,13.082723,9.585747,...,8.799330,6.381569,8.666479,10.726714,10.091333,8.735554,9.614992,8.725782,8.461880,10.144518


In [ ]:
mixture.to_csv(OUTPUT_FILENAME, sep="\t")
print(f"Saved CIBERSORTx-ready mixture file to: {OUTPUT_FILENAME}")

# Quick sanity check: preview the raw text format CIBERSORTx will actually read
with open(OUTPUT_FILENAME) as f:
    for i, line in enumerate(f):
        print(line.strip()[:150])
        if i > 3:
            break


Saved CIBERSORTx-ready mixture file to: /content/cibersortx_input/GSE14905_cibersortx_mixture_healthy_vs_lesional.txt
GeneSymbol	GSM372286	GSM372287	GSM372288	GSM372289	GSM372290	GSM372291	GSM372292	GSM372293	GSM372294	GSM372295	GSM372296	GSM372297	GSM372298	GSM372299
A1BG	4.843981345074605	8.08649976295792	4.594892444974296	4.595631374673704	4.590429434439752	4.576177001325383	4.603877114690559	4.628442823390092	4.
A1BG-AS1	7.193065114420347	12.779943729907854	7.183169440516026	8.95727235832725	7.159781165311189	8.342396952697577	7.193715747776747	7.4253449347835
A1CF	5.918883203802557	5.8798278978925484	6.170652674210569	6.137120496029627	5.903609810661325	6.891795948314288	6.08227256850248	6.300250502467272	7
A2M	1020.306972199247	1265.2684743777509	984.615163623961	2355.413795303804	670.1771677327868	505.435006221239	589.9193186088147	459.3033344991479	137


In [ ]:
from google.colab import files
files.download(OUTPUT_FILENAME)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Before you upload to CIBERSORTx

- Double-check the printed "log2 vs linear" decision in Section 3 against what you know about GSE14905 --
  if you have reason to believe the values are already linear-scale MAS5 calls rather than log2 RMA,
  override the auto-detection.
- **This version excludes Uninvolved Skin (non-lesional) samples by default** -- the mixture file should
  contain 21 healthy + 33 lesional = 54 samples (or fewer, if you also set `EXCLUDE_GSM_IDS`). Check the
  printed sample count in Section 4 matches what you expect before uploading.
- LM22 uses HGNC gene symbols; this notebook uses whatever symbol format GPL570's annotation table
  provides, which is normally HGNC-compatible, but worth a quick spot-check on a handful of overlapping
  gene names between your file and LM22 before trusting the run.
- CIBERSORTx recommends running in **B-mode batch correction** if your mixture and signature matrix come
  from different profiling platforms (which is the case here: LM22 is microarray-derived, GSE14905 is also
  microarray, so platform mismatch is less of a concern than if you were mixing in RNA-seq -- but still
  worth selecting B-mode on the web portal since it's the safer default).


In [ ]:
print("Conditions present in the filtered data:")
print(sample_meta["condition"].value_counts())
print(f"\nTotal samples in the final mixture file: {mixture.shape[1]}")

Conditions present in the filtered data:
condition
lesional    33
healthy     21
Name: count, dtype: int64

Total samples in the final mixture file: 54


In [ ]:
mixture_file_path = "/content/cibersortx_input/GSE14905_cibersortx_mixture_healthy_vs_lesional.txt"

# Read the file to verify its contents
try:
    mixture_df = pd.read_csv(mixture_file_path, sep='\t', index_col=0)
    num_samples_in_file = mixture_df.shape[1]
    print(f"The file '{mixture_file_path}' contains {num_samples_in_file} samples.")
    if num_samples_in_file == 54:
        print("This confirms that the file contains only 54 samples.")
    else:
        print(f"Warning: The file contains {num_samples_in_file} samples, which is not 54.")
except FileNotFoundError:
    print(f"Error: The file '{mixture_file_path}' was not found.")
except Exception as e:
    print(f"An error occurred while reading the file: {e}")

The file '/content/cibersortx_input/GSE14905_cibersortx_mixture_healthy_vs_lesional.txt' contains 54 samples.
This confirms that the file contains only 54 samples.
